In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import tqdm
import os
import numpy as np
import zipfile
import torch
import torch.optim as optim
import random

from huggingface_hub import snapshot_download

from data import ShapeNetClassficationDataset, ShapeNetSegmentationDataset
from PointNet import PointNetCls, PointNetSeg, PointNetClsTrainer, PointNetSegTrainer

In [ ]:
# shapenetcore_partanno_segmentation

data_root = '../../Data/shapenetcore_partanno_segmentation'
repo_id = "wangps/shapenet_segmentation"

snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    local_dir=data_root,          
    local_dir_use_symlinks=False,
    #allow_patterns=[],     
)

for file in os.listdir(data_root):
    if file.endswith('.zip'):                     
        zip_path = os.path.join(data_root, file)
        extract_dir = os.path.join(data_root, os.path.splitext(file)[0])
        if os.path.exists(extract_dir):
            continue  
        os.makedirs(extract_dir, exist_ok=True)
        print(f"unziping {file}")   
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)     

data_dir=os.path.join(data_root,"shapenetcore_partanno_segmentation_benchmark_v0_normal","shapenetcore_partanno_segmentation_benchmark_v0_normal")

In [ ]:
random.seed(0)
torch.manual_seed(0)

BATCH_SIZE=32
NUM_WORKERS=1

model_save_root = "../../Models/PointNet"
os.makedirs(model_save_root,exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

# PointNet

### Classification 

In [ ]:
train_dataset = ShapeNetClassficationDataset(
        root=data_dir,
        split='train',
        npoints=1024,
        with_data_augmentation=True)

test_dataset = ShapeNetClassficationDataset(
        root=data_dir,
        split='test',
        npoints=1024,
        with_data_augmentation=False)

train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=int(NUM_WORKERS))

test_dataloader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=int(NUM_WORKERS))

print(len(train_dataset), len(test_dataset))
num_classes = len(train_dataset.classes)
print('classes', num_classes)

In [ ]:
classifier = PointNetCls(k=num_classes)
optimizer = optim.Adam(classifier.parameters(), lr=0.01, betas=(0.9, 0.999))
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.8)

trainer=PointNetClsTrainer(
    model=classifier,
    optimizer=optimizer,
    scheduler=scheduler,
    dtype=float,
    device=DEVICE
)

trainer.train(
    num_epochs=2,
    train_dataloader=train_dataloader,
    log_interval=10
)

torch.save(trainer.model.state_dict(), os.path.join(model_save_root,"PointNetCls.pth"))

In [ ]:
classifier = PointNetCls(k=num_classes).to(DEVICE)
state_dict=torch.load(os.path.join(model_save_root,"PointNetCls.pth"), map_location=DEVICE)
classifier.load_state_dict(state_dict)
classifier.eval()

total_correct = 0
total_testset = 0

for data in tqdm.tqdm(test_dataloader):
    points, target = data
    points=points.to(DEVICE)
    target=target.to(DEVICE)
    target = target[:, 0]

    pred, _ = classifier(points)
    pred_choice = pred.data.max(1)[1]

    correct = pred_choice.eq(target.data).cpu().sum()
    
    total_correct += correct.item()
    total_testset += points.size()[0]

print("final accuracy {}".format(total_correct / float(total_testset)))

### Segmentation

In [ ]:
train_dataset = ShapeNetSegmentationDataset(
    root=data_dir,
    npoints = 1024,
    class_choice=["Airplane"])
train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=int(NUM_WORKERS))

test_dataset = ShapeNetSegmentationDataset(
    root=data_dir,
    npoints = 1024,
    class_choice=["Airplane"],
    split='test',
    with_data_augmentation=False)
test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=int(NUM_WORKERS))

print(len(train_dataset), len(test_dataset))
num_classes = train_dataset.num_seg_classes
print('classes', num_classes)

In [ ]:
classifier = PointNetSeg(k=num_classes)
optimizer = optim.Adam(classifier.parameters(), lr=0.001, betas=(0.9, 0.999))
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

trainer=PointNetSegTrainer(
    model=classifier,
    optimizer=optimizer,
    scheduler=scheduler,
    dtype=float,
    device=DEVICE
)
trainer.train(
    num_epochs=2,
    train_dataloader=train_dataloader,
    log_interval=10
)

torch.save(trainer.model.state_dict(), os.path.join(model_save_root,"PointNetSeg.pth"))

In [ ]:
classifier = PointNetSeg(k=num_classes).to(DEVICE)
state_dict=torch.load(os.path.join(model_save_root,"PointNetSeg.pth"), map_location=DEVICE)
classifier.load_state_dict(state_dict)
classifier.eval()


## benchmark mIOU
shape_ious = []
for data in tqdm.tqdm(test_dataloader):
    points, target = data
    points=points.to(DEVICE)
    target=target.to(DEVICE)

    pred = classifier(points)
    pred_choice = pred.data.max(2)[1]

    pred_np = pred_choice.cpu().data.numpy()
    target_np = target.cpu().data.numpy() - 1

    for shape_idx in range(target_np.shape[0]):
        parts = range(num_classes)#np.unique(target_np[shape_idx])
        part_ious = []
        for part in parts:
            I = np.sum(np.logical_and(pred_np[shape_idx] == part, target_np[shape_idx] == part))
            U = np.sum(np.logical_or(pred_np[shape_idx] == part, target_np[shape_idx] == part))
            if U == 0:
                iou = 1 #If the union of groundtruth and prediction points is empty, then count part IoU as 1
            else:
                iou = I / float(U)
            part_ious.append(iou)
        shape_ious.append(np.mean(part_ious))

print("mIOU for class Airplane: {}".format(np.mean(shape_ious)))